<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">

# Python & AI for Rates, Bonds, and Credit

## Appendix F · Numerical Methods for Fixed Income

[Open in Google Colab](https://colab.research.google.com/github/yhilpisch/ratescode/blob/main/notebooks/appx_f_numerical_methods_fixed_income.ipynb)

&copy; Dr. Yves J. Hilpisch<br>
AI-Powered by different LLMs<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh

## Learning goals

Most fixed-income calculations end in a numerical method rather than a closed form. After working through this companion, you can:

- explain how bisection brackets a bond yield and why bracketing is more robust than Newton's method;
- price a continuous annuity with the trapezoid rule and quantify how the quadrature error shrinks with the step size;
- choose finite-difference bump sizes that avoid both curvature contamination and rounding noise;
- state how Monte Carlo standard error scales with the path count and report simulation results with uncertainty.

## Prepare the companion repository

This setup cell works with a local checkout or a fresh Google Colab runtime. In Colab it clones the public companion repository, then sets `ROOT` so the chapter code and frozen data snapshots are found without manual path edits.

In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

REPO_URL = "https://github.com/yhilpisch/ratescode.git"
candidates = [Path.cwd(), Path.cwd().parent,
              Path("/content/07_ratescode")]
ROOT = next((p.resolve() for p in candidates
             if (p / "code").is_dir()
             and (p / "data").is_dir()), None)
if ROOT is None:
    ROOT = Path("/content/07_ratescode")
    subprocess.run(["git", "clone", "--depth", "1",
                    REPO_URL, str(ROOT)], check=True)

packages = {"numpy": "numpy", "pandas": "pandas",
            "matplotlib": "matplotlib"}
missing = [package for module, package in packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install",
                    *missing], check=True)
print(f"Repository: {ROOT}")
print(f"Data files: {len(list((ROOT / 'data').glob('*.csv')))}")


## Connect the appendix to a calculation

Chapter 6 bootstraps curves by solving one root-finding problem per instrument, Chapter 22 estimates value at risk and expected shortfall from simulated loss distributions, and Chapter 23 reprices calibrated models as a verification step. This appendix collects the numerical ideas those chapters rely on and sets tolerances by financial materiality rather than machine precision alone.

The companion figure script `appx_f_numerical_convergence.py` contrasts the two recurring convergence patterns: a root bracket that halves every bisection iteration and a Monte Carlo standard error that shrinks only with the square root of the path count.

In [ ]:
import pandas as pd
figure_scripts = sorted((ROOT / "code" / "figures").glob("appx_f_*.py"))
pd.DataFrame({"figure_script": [p.name for p in figure_scripts]})

## Review prompts

- Write the pricing error function for a semiannually compounded bond yield and state what makes bisection a bracketing method.
- For the mini-lab, predict the trapezoid error at 32 steps before running it, and state the rule you used.
- Explain why a one-basis-point bump is usually a stable DV01 estimate while a ten-basis-point bump shows curvature contamination.
- State the path count needed to cut a Monte Carlo standard error of 0.090 to about 0.009, and the convergence rate that forces this cost.

## Mini-lab: Trapezoid rule versus a closed-form annuity

The mini-lab prices the present value of one currency unit paid continuously over five years at a flat 3% continuously compounded zero rate, which equals the integral of exp(-0.03 t) from 0 to 5. The trapezoid rule approximates the same integral on uniform grids of 4, 8, and 16 steps, so the step sizes are 1.25, 0.625, and 0.3125 years. The comparison isolates one mechanism: the trapezoid error is proportional to the square of the step size. All inputs are illustrative.

In [ ]:
import numpy as np
import pandas as pd

rate, T = 0.03, 5.0  # zero rate and horizon, both annual
exact = (1 - np.exp(-rate * T)) / rate  # annuity value in closed form

def trapezoid(steps):
    grid = np.linspace(0, T, steps + 1)  # quadrature nodes
    values = np.exp(-rate * grid)  # discount factors on the grid
    h = T / steps  # step size
    return h * (values[0] / 2 + values[1:-1].sum() + values[-1] / 2)

errs = {n: trapezoid(n) - exact for n in (4, 8, 16)}  # signed errors
pd.Series({"exact": exact, "trap_4": trapezoid(4),
           "trap_8": trapezoid(8), "trap_16": trapezoid(16),
           "error_ratio_4_to_8": errs[4] / errs[8]})

### What to observe

The exact annuity value is 4.643067. The trapezoid estimates are 4.643612, 4.643203, and 4.643101, with signed errors of 5.44e-4, 1.36e-4, and 3.40e-5. Halving the step size divides the error by four, and the displayed error ratio of 4.00 confirms the quadratic convergence, so a modest grid already reaches pricing accuracy.

## Exercises

1. **Sharpen the grid.** Add 32 and 64 steps to the mini-lab and verify that the error ratio stays near 4 for every halving. *Hint:* Extend the tuple of step counts in the error dictionary.
2. **Bisection sequence.** Reproduce the appendix's five-step bisection table for a bond whose price is 1.42 above the market price at a 4.00% yield and 0.31 below it at 4.50%. State the bracket after five steps and the best yield estimate. *Hint:* The midpoints are 4.00%, 4.50%, 4.25%, 4.375%, and 4.4375%, leaving the bracket 4.375% to 4.4375%.
3. **DV01 bump check.** Compute the central-difference sensitivity of the mini-lab annuity to the 3% rate with bumps of 10, 1, and 0.1 basis points, and compare with the analytic derivative. *Hint:* Differentiate (1 minus exp of minus rT) over r with respect to r; note which bump drifts first as curvature grows.
4. **Monte Carlo budget.** Estimate how many paths are needed to bring a payoff-estimate standard error from 0.090 below 0.010, and state the convergence law that sets the cost. *Hint:* The error scales with one over the square root of the path count, so 100,000 paths give 0.009.
5. **Tolerance choice.** A curve bootstrap uses a yield tolerance of 1e-12 while the desk reports to the nearest dollar. Argue for a more material tolerance and one risk of loosening it too far. *Hint:* Tie the tolerance to the smallest economically meaningful price difference in the instrument set.

For more detail, see Appendix F of *Python & AI for Rates, Bonds, and Credit*.

<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">